用于记录或测试一些可以复用的代码

导入

In [ ]:
import numpy as np
import pandas as pd
import glob
import os.path as osp

数据集预览

In [ ]:
from datasets_cfg import *
kind="ABCD"
dataset = datasets[kind]
labeldf = pd.read_csv(dataset["demographics_path"])
# # if ABCD
# labeldf = labeldf[labeldf["eventname"]=="baseline_year_1_arm_1"]
# print(labeldf.columns[labeldf.columns.str.contains('bir', case=False)])

# tgt_labels = labeldf.filter(items=dataset["tgt_label_list"])
# labeldf = labeldf.dropna(axis=1, thresh=len(labeldf)*0.75)
# labeldf.columns

Dataset类测试

In [ ]:
from datasets_cfg import datasets
from datasets_base import load_data

# Use predefined paths from datasets_cfg
dataset_keys = ["HCD", "S1200", "ABCD"]
dt_name = "S1200"
dt_cfg = datasets["S1200"]

In [ ]:
dataset = load_data(
    conn_dir=dt_cfg["conn_dir"],
    atlas_name=["bna246"],
    conn_type='SC',
    conn_kind=['fiber_length'],
    dt_name=dt_name,
    dt_cfg=dt_cfg
)

In [ ]:
dataset.keys()

In [ ]:
len(dataset["scores"])

In [ ]:
dataset["scores"]["CogTotalComp_AgeAdj"].values.shape

In [ ]:
dataset["nets"]["bna246"]["connectome_mean_FA_10M.csv"].shape

In [ ]:
graphs = dataset["nets"]["bna246"]["connectome_mean_FA_10M.csv"]
scores = dataset["scores"]["CogTotalComp_AgeAdj"].values

机器学习预测+haufe变换测试

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import edge_selection
from edge_selection import rearrange_edges, select_sig_edges, haufe_transform
from models import get_regression_model

In [ ]:
def run_one_dataset(out_dir,dataset_name, mats_subj_roi_roi, labels, pred_label_type="label",RANDOM_STATE=42):
    # 输入格式转换为 edge_selection 所需：num_roi*num_roi*num_subj
    edges_roi_roi_subj = np.transpose(mats_subj_roi_roi, (1, 2, 0))
    num_subj = mats_subj_roi_roi.shape[0]
    num_roi = mats_subj_roi_roi.shape[1]

    # 步骤1：重组 edge（按需求先调用）
    _all_edges = rearrange_edges(edges_roi_roi_subj, num_roi, num_subj)

    # select_sig_edges 依赖全局 pcorr_res，这里提前创建
    edge_selection.pcorr_res = np.zeros((num_roi, num_roi, 2), dtype=float)

    # 步骤2：选择关键 edge
    train_picked_edges = select_sig_edges(labels, edges_roi_roi_subj, num_roi, measurement="pcorr")

    # sklearn 输入格式：样本在前
    X = train_picked_edges.T
    y = labels.astype(float)

    idx = np.arange(num_subj)
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, idx, test_size=0.25, random_state=RANDOM_STATE
    )

    methods = [
        {"method_type": "linear", "method_name": "lasso", "params": {"alpha": 0.05, "max_iter": 5000}},
        {"method_type": "linear", "method_name": "ridge", "params": {"alpha": 1.0}},
        {"method_type": "linear", "method_name": "linear", "params": {}},
        {"method_type": "linear", "method_name": "huber", "params": {"epsilon": 1.35}},
        {"method_type": "nonlinear", "method_name": "kernel_ridge_rbf", "params": {"alpha": 1.0, "gamma": 0.05}},
        {"method_type": "nonlinear", "method_name": "decision_tree", "params": {"max_depth": 6, "random_state": RANDOM_STATE}},
        {"method_type": "nonlinear", "method_name": "gradient_boosting", "params": {"random_state": RANDOM_STATE}},
        {"method_type": "mlp", "method_name": "single_layer_mlp", "params": {"hidden_layer_sizes": (64,), "max_iter": 500, "random_state": RANDOM_STATE}},
    ]

    per_subject_rows = []
    summary_rows = []

    for m in methods:
        model = get_regression_model(m["method_type"], m["method_name"], **m["params"])
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # 预测显著性：预测值与真实值的 Pearson 相关及 p 值
        if np.std(y_pred) > 1e-12 and np.std(y_test) > 1e-12:
            pred_r, pred_p = pearsonr(y_test, y_pred)
        else:
            pred_r, pred_p = np.nan, np.nan

        # pred_effiency 这里用 R2（最常见的回归预测效力指标）
        pred_efficiency = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        # Haufe 贡献：仅当模型有 coef_ 时直接计算
        haufe_csv = ""
        haufe_status = "not_supported"
        haufe_error = ""
        if hasattr(model, "coef_"):
            try:
                model_coef = np.asarray(model.coef_).reshape(-1)
                haufe_mat = haufe_transform(train_picked_edges, y, model_coef)
                haufe_csv = os.path.join(out_dir, f"haufe_{dataset_name}_{m['method_type']}_{m['method_name']}.csv")
                pd.DataFrame(haufe_mat).to_csv(haufe_csv, index=False)
                haufe_status = "ok"
            except Exception as e:
                haufe_status = "failed"
                haufe_error = str(e)

        for local_i, subj_i in enumerate(idx_test):
            per_subject_rows.append({
                "dataset": dataset_name,
                "subject_index": int(subj_i),
                "method_type": m["method_type"],
                "method_name": m["method_name"],
                "pred_label_type": pred_label_type,
                "true_label": float(y_test[local_i]),
                "pred_scores": float(y_pred[local_i]),
                "pred_effiency": float(pred_efficiency),
            })

        summary_rows.append({
            "dataset": dataset_name,
            "method_type": m["method_type"],
            "method_name": m["method_name"],
            "pred_label_type": pred_label_type,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            "pred_effiency_r2": float(pred_efficiency),
            "mae": float(mae),
            "rmse": float(rmse),
            "pearson_r": float(pred_r) if not np.isnan(pred_r) else np.nan,
            "pearson_p": float(pred_p) if not np.isnan(pred_p) else np.nan,
            "haufe_status": haufe_status,
            "haufe_csv": haufe_csv,
            "haufe_error": haufe_error,
        })

    return pd.DataFrame(per_subject_rows), pd.DataFrame(summary_rows)


In [ ]:
sc_pred_df, sc_summary_df = run_one_dataset(out_dir="./res_data",
                                            dataset_name="SC", 
                                            mats_subj_roi_roi=graphs, 
                                            labels=scores, 
                                            pred_label_type="CogTotalComp_AgeAdj")

In [ ]:
haufe_res_exm = pd.read_csv("res_data/haufe_SC_linear_lasso.csv").values

In [ ]:
haufe_res_exm.shape

绘制haufe分布

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nilearn import datasets
import nibabel as nib

from draw_brain import draw_connectome, draw_atlas_roi, get_coords, compute_roi_values

In [ ]:
# atlas_coords = get_coords(nib.load("./BN_Atlas_246_1mm.nii.gz"))
atlas_coords = np.load("./atlas_coords.npy")

In [ ]:
display_conn = draw_connectome(
    adjacency_matrix=haufe_res_exm,
    coords=atlas_coords,
    edge_threshold="99.9%",
    title="BNA Subset Connectome",
    save_dir="pics",
    filename="demo_connectome_bna.png",
    # edge_vmin=np.min(haufe_res_exm),
    # edge_vmax=0,
    node_color="none",
)
plt.show()

In [ ]:
roi_values = compute_roi_values(haufe_res_exm)

In [ ]:
max(roi_values),min(roi_values)

In [ ]:
value_img, display_roi = draw_atlas_roi(
    roi_values=roi_values,
    atlas_img="./BN_Atlas_246_1mm.nii.gz",
    label_ids=range(0,246),
    view="surf",
    # threshold=np.min(roi_values),
    title="BNA Subset ROI Values (Surface)",
    save_dir="pics",
    filename="demo_atlas_roi_bna_surf.png",
)

HCD批量计算：label x atlas x SC type 的拟合、Haufe 变换与绘图

In [ ]:
from datasets_base import SC_KIND
from batch_runner import run_hcd_batch

HCD_ATLASES = ["aal116", "bna246", "schaefer200_s1"]
HCD_SC_TYPES = list(SC_KIND.keys())

hcd_pred_df, hcd_summary_df = run_hcd_batch(
    dataset_name="S1200",
    atlas_names=HCD_ATLASES,
    sc_types=HCD_SC_TYPES,
    label_names=None,
    random_state=42,
)

hcd_summary_df

In [ ]:
hcd_summary_df.sort_values(["pred_label_type", "atlas", "sc_type", "method_name"]).head()